# Compare curve vs. block representation: stats only

Assumes every experiment already has **three** trained ensembles under `MODEL_ROOT`, each with a `stats.pkl` saved alongside it (i.e. `cnnpz.get_all_stats(..., save=True, saveroot=save_dir + "/stats.pkl")` was run once for each):

- the block-representation baseline (`*_ensemble_CNN_6layers`, no `curve` tag)
- the 8-band curve representation (`*_curve_ensemble_CNN_6layers`)
- the 9-band curve representation, with Roman Y106 brought back (`*_curve_9band_ensemble_CNN_6layers`)

This notebook does no data loading, no filter-bank/block reconstruction, and no model inference -- it just reads the saved `(stats, redshift_stats, imag_stats)` pickles with `cnnpz.read_stats` and compares them.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cnnpz

cnnpz.set_plot_style()

In [ ]:
# Parametric paths, same convention as CNN_photoz-ensemble-cardinal.ipynb
USER = os.environ.get("USER", "jaimerz")
PSCRATCH = os.environ.get("PSCRATCH", f"/pscratch/sd/{USER[0]}/{USER}")

MODEL_ROOT = os.path.join(PSCRATCH, "cnnpz", "noisy_Cardinal", "models")
# pre-training/fine-tuning models were saved under a different root in the source notebook
PRETRAIN_MODEL_ROOT = "/pscratch/sd/q/qhang/cnnpz/noisy_Cardinal/models"

# (experiment name, root, block subdir, curve (8-band) subdir, curve (9-band) subdir)
experiments = [
    ("Y1 complete", MODEL_ROOT, "y1_complete_ensemble_CNN_6layers", "y1_complete_curve_ensemble_CNN_6layers", "y1_complete_curve_9band_ensemble_CNN_6layers"),
    ("Y1 NIR-dropout", MODEL_ROOT, "y1_misnir_ensemble_CNN_6layers", "y1_misnir_curve_ensemble_CNN_6layers", "y1_misnir_curve_9band_ensemble_CNN_6layers"),
    ("Y10 complete", MODEL_ROOT, "y10_complete_ensemble_CNN_6layers", "y10_complete_curve_ensemble_CNN_6layers", "y10_complete_curve_9band_ensemble_CNN_6layers"),
    ("Y10 NIR-dropout", MODEL_ROOT, "y10_misnir_ensemble_CNN_6layers", "y10_misnir_curve_ensemble_CNN_6layers", "y10_misnir_curve_9band_ensemble_CNN_6layers"),
    ("Y1 spec-select", MODEL_ROOT, "y1_specsel_ensemble_CNN_6layers", "y1_specsel_curve_ensemble_CNN_6layers", "y1_specsel_curve_9band_ensemble_CNN_6layers"),
    ("Y10 spec-select", MODEL_ROOT, "y10_specsel_ensemble_CNN_6layers", "y10_specsel_curve_ensemble_CNN_6layers", "y10_specsel_curve_9band_ensemble_CNN_6layers"),
    ("Y10 pre-training", PRETRAIN_MODEL_ROOT, "y10_pretrain_ensemble_CNN_6layers", "y10_pretrain_curve_ensemble_CNN_6layers", "y10_pretrain_curve_9band_ensemble_CNN_6layers"),
    ("Y10 fine-tune", PRETRAIN_MODEL_ROOT, "y10_finetune_ensemble_CNN_6layers", "y10_finetune_curve_ensemble_CNN_6layers", "y10_finetune_curve_9band_ensemble_CNN_6layers"),
]

In [ ]:
# same binning used everywhere else in the project when these stats were generated
redshift_bins = np.linspace(0, 2.5, 11)
imag_bins = np.linspace(18, 25.5, 11)

results = []

for name, root, block_subdir, curve_subdir, curve9_subdir in experiments:
    block_stats, block_z_stats, block_i_stats = cnnpz.read_stats(os.path.join(root, block_subdir, "stats.pkl"))
    curve_stats, curve_z_stats, curve_i_stats = cnnpz.read_stats(os.path.join(root, curve_subdir, "stats.pkl"))
    curve9_stats, curve9_z_stats, curve9_i_stats = cnnpz.read_stats(os.path.join(root, curve9_subdir, "stats.pkl"))

    print(f"=== {name} ===")
    print(cnnpz.stats_to_markdown(block_stats, curve_stats, data_title=("block", "curve_8band")))
    print(cnnpz.stats_to_markdown(block_stats, curve9_stats, data_title=("block", "curve_9band")))

    cnnpz.compare_binned_stats(redshift_bins, imag_bins, block_z_stats, block_i_stats, curve_z_stats, curve_i_stats)
    plt.suptitle(f"{name} -- block vs. curve (8-band)")
    plt.show()

    cnnpz.compare_binned_stats(redshift_bins, imag_bins, block_z_stats, block_i_stats, curve9_z_stats, curve9_i_stats)
    plt.suptitle(f"{name} -- block vs. curve (9-band)")
    plt.show()

    # stats tuple is (mean, mean_err, std, outlier_rate, abs_outlier_rate) -- std is biweight sigma_z
    sigma_block, sigma_curve, sigma_curve9 = block_stats[2], curve_stats[2], curve9_stats[2]
    sigmas = {"block": sigma_block, "curve_8band": sigma_curve, "curve_9band": sigma_curve9}
    winner = min(sigmas, key=sigmas.get)
    results.append({
        "experiment": name,
        "sigma_z_block": sigma_block,
        "sigma_z_curve_8band": sigma_curve,
        "sigma_z_curve_9band": sigma_curve9,
        "outlier_rate_block": block_stats[3],
        "outlier_rate_curve_8band": curve_stats[3],
        "outlier_rate_curve_9band": curve9_stats[3],
        "winner (lowest sigma_z)": winner,
    })

In [ ]:
summary = pd.DataFrame(results)
win_counts = summary["winner (lowest sigma_z)"].value_counts()
print(summary.to_string(index=False))
print(f"\nwins by representation (lowest biweight sigma_z) out of {len(summary)} experiments:")
print(win_counts.to_string())
summary